#### 📊 Consultas Analíticas

##### Objetivo
Conjunto de **consultas pré-construídas** que respondem perguntas estratégicas de negócio, aproveitando o modelo Star Schema para queries otimizadas.

##### Filtro de Qualidade
Todas as consultas aplicam:
```sql
WHERE
  quantidade IS NOT NULL AND quantidade > 0
  AND valor_unitario > 0
  AND custo_unitario > 0
```
Garante que apenas vendas válidas sejam analisadas.

##### Questões de Negócio
1. 📈 **Evolução Temporal**: Como estão as tendências ao longo do tempo?
2. 📅 **Sazonalidade**: Quais períodos tem melhor/pior performance?
3. 🏪 **Performance por Loja**: Qual o desempenho de lojas e grupos?
4. 🔁 **Comparação Anual**: Como 2024 vs 2025?
5. 📊 **Volume de Vendas**: Onde está concentrado o volume?

In [0]:
%sql
USE vendas_pecas.camada_ouro;

### 📈 Questão 1: Evolução Temporal

#### Como evoluíram nosso faturamento e resultados ao longo do tempo?

**Análise Mês a Mês:**
* Faturamento mensal
* Lucro mensal
* Margem de lucro (%)
* Quantidade de peças vendidas

In [0]:
%sql
--separa em outro notebook

SELECT
  ano_venda AS ano,
  mes_venda AS mes,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro,
  SUM(lucro) / SUM(faturamento) * 100 AS margem_lucro,
  SUM(quantidade) AS quantidade_pecas
FROM 
  tb_fato_vendas fv
JOIN
  tb_dim_tempo dv ON fv.sk_data_venda = dv.sk_data_venda
WHERE
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0 and
  ano_venda BETWEEN 2024 and 2025
GROUP BY 
  mes, ano
ORDER BY 
  ano_venda, mes_venda

In [0]:
%sql
SELECT
  ano_venda AS ano,
  mes_venda AS mes,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro,
  SUM(lucro) / SUM(faturamento) * 100 AS margem_lucro,
  SUM(quantidade) AS quantidade_pecas
FROM 
  tb_vendas_obt
WHERE
  quantidade > 0 and
  valor_unitario > 0 and
  custo_unitario > 0 and
  ano_venda BETWEEN 2024 and 2025
GROUP BY 
  mes, ano
ORDER BY 
  ano_venda, mes_venda

### 📅 Questão 2: Sazonalidade e Periodicidade

#### Quais períodos apresentam melhor e pior performance?

**Análises Disponíveis:**

1. **Semestral** (2024-2025):
   * Comparação 1º vs 2º semestre
   * Identifica padrões semestrais

2. **Mensal** (2024-2025):
   * Performance mês a mês
   * Identifica melhores/piores meses

3. **Quinzenal** (2024):
   * Granularidade: 1ª vs 2ª quinzena do mês
   * Identifica padrões intra-mensais

In [0]:
%sql
SELECT 
  ano_venda,
  CASE
    WHEN mes_venda <= 6 
      THEN '1º Semestre'
    WHEN mes_venda > 6 
      THEN '2º Semestre'
  END as semestre,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro,
  SUM(quantidade) AS quantidade_pecas
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_tempo ddt ON fv.sk_data_venda = ddt.sk_data_venda
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0 AND
  ano_venda BETWEEN 2024 and 2025
GROUP BY 
  ano_venda, semestre
ORDER BY 
  ano_venda

In [0]:
%sql
SELECT 
  ano_venda,
  CASE
    WHEN mes_venda <= 6 
      THEN '1º Semestre'
    WHEN mes_venda > 6 
      THEN '2º Semestre'
  END as semestre,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro,
  SUM(quantidade) AS quantidade_pecas
FROM 
  tb_vendas_obt
WHERE
  quantidade IS NOT NULL and
  quantidade > 0 and
  valor_unitario > 0 and
  custo_unitario > 0
GROUP BY 
  ano_venda, semestre
ORDER BY 
  ano_venda

In [0]:
%sql
SELECT 
    ano_venda, 
    mes_venda, 
    SUM(faturamento) AS faturamento, 
    SUM(lucro) as lucro,
    SUM(quantidade) AS quantidade_pecas
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_data_venda ddt on fv.sk_data_venda = ddt.sk_data_venda
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY  
  ano_venda, mes_venda
ORDER BY 
  ano_venda, mes_venda

In [0]:
%sql
SELECT 
  ano_venda,
  mes_venda,
  CASE
    WHEN day(data_venda) <= 15 
      THEN '1º Quinzena'
    WHEN day(data_venda) > 15 
      THEN '2º Quinzena'
  END AS Semestre,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro,
  SUM(quantidade) AS quantidade_pecas
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_data_venda ddt ON fv.sk_data_venda = ddt.sk_data_venda
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
  ano_venda, mes_venda, semestre
ORDER BY 
  ano_venda,mes_venda

### 🏪 Questão 3: Desempenho por Localidade

#### Qual o desempenho das vendas por loja, grupo de lojas e vendedor?

**Métricas de Desempenho:**
* Faturamento total
* Lucro total
* Margem de lucro (%)
* Ticket médio (faturamento / nº vendas)
* Lucro médio unitário

**Análises Disponíveis:**

1. **Por Grupo de Loja e Loja Individual**:
   * Hierarquia: Grupo → Loja
   * Ranking de performance
   * Identifica melhores/piores unidades

2. **Por Vendedor**:
   * Performance individual
   * Rankings de vendedores
   * Base para comissões e incentivos

In [0]:
%sql
SELECT 
  grupo_loja,
  loja_nome, 
  SUM(faturamento) AS faturamento, 
  SUM(lucro) AS lucro,
  SUM(lucro) / SUM(faturamento) * 100 AS margem_lucro,
  SUM(faturamento) / count(IdNotaFiscal) AS ticket_medio,
  SUM(lucro) / SUM(quantidade) AS lucro_medio_unitario
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_loja dl ON fv.sk_loja = dl.sk_loja
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
   grupo_loja, loja_nome
ORDER BY 
  faturamento desc

In [0]:
%sql
SELECT
  vendedor_nome,
  SUM(faturamento) AS faturamento,
  SUM(lucro)AS lucro,
  SUM(lucro) / SUM(faturamento) * 100 AS margem_lucro,
  SUM(faturamento) / count(IdNotaFiscal) AS ticket_medio,
  SUM(lucro) / SUM(quantidade) AS lucro_medio_unitario
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_vendedor dv ON fv.sk_vendedor = dv.sk_vendedor
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0  and
  dv.sk_vendedor = fv.sk_vendedor

GROUP BY 
  vendedor_nome
ORDER BY 
  faturamento DESC
   

### 🔁 Questão 4: Comparação entre Anos

#### Como está o comparativo entre anos?

**Métricas Comparadas (2024 vs 2025):**
* Faturamento total
* Lucro total
* Custo total
* Quantidade de peças vendidas
* Número de vendas

In [0]:
%sql
SELECT
    ano_venda AS ano,
    SUM(faturamento) AS faturamento, 
    SUM(lucro) AS lucro,  
    SUM(custo_unitario * quantidade) AS custo_total,
    SUM(quantidade) AS qtd_pecas_vendidas,
    COUNT(IdNotaFiscal) AS n_vendas,
    ROW_NUMBER() OVER(
      PARTITION BY ano_venda
      ORDER BY sum(faturamento) DESC
  ) AS rn_quantidade
  FROM 
    tb_fato_vendas fv
  JOIN 
    tb_dim_data_venda ddt ON ddt.sk_data_venda = fv.sk_data_venda
  JOIN
    tb_dim_produtos dp ON dp.sk_produto = fv.sk_produto
  WHERE
    fv.quantidade IS NOT NULL and
    fv.quantidade > 0 and
    fv.valor_unitario > 0 and
    fv.custo_unitario > 0
  GROUP BY  ano

### 📊 Questão 5: Concentração de Volume

#### Quais unidades e grupos concentram maior volume de vendas?

**Objetivo:**
Identificar onde está concentrado o **volume de transações** (número de vendas) versus o **volume de produtos** (quantidade vendida).

**Métricas:**
* Contagem de vendas por grupo/loja
* Quantidade de peças vendidas
* Distribuição percentual


In [0]:
%sql
SELECT 
  grupo_loja,
  loja_nome,
  count(IdNotaFiscal) AS n_vendas,
  SUM(quantidade) AS qtd_vendida
FROM
  tb_fato_vendas fv
JOIN 
  tb_dim_loja dl ON dl.sk_loja = fv.sk_loja
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0  
GROUP BY 
   grupo_loja, loja_nome
ORDER BY n_vendas DESC


In [0]:
%sql
SELECT 
  grupo_loja,
  count(IdNotaFiscal) AS n_vendas,
  SUM(quantidade) AS qtd_vendida
FROM
  tb_fato_vendas fv
JOIN 
  tb_dim_loja dl ON dl.sk_loja = fv.sk_loja
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
  grupo_loja
ORDER BY n_vendas DESC